# STGAN ConvGRU su ERA5: cosa trova il modello

Run `stgan_convgru_fullgrid_recent16_cyclic_bf16_cleanD_wdaily_seasonal_trend_ep3`: 3 epoche, training
1980–2004, test 2005–2025. Ha tutte le aggiunte: griglia intera, storia di 16 passi, fix del dropout del
discriminatore, normalizzazione stagionale con trend, pesi delle variabili `daily_change`, MC Dropout con 20
campioni.

Lo score del modello viene confrontato con due **calcoli semplici**, che non usano nessun modello:
- **distanza dal normale**: quanto il valore si allontana dal normale stagionale della sua cella, giorno e ora;
- **variazione in 3 ore**: quanto il valore è cambiato rispetto all'istante prima.

I tre score sono letti dopo la climatologia, cioè standardizzati per cella, mese e ora. Un valore
(cella × istante) è **segnalato** da uno score se è nel suo `TOP_PERCENT` più alto.

Il notebook legge solo i file della run, senza GPU, su un istante ogni `STRIDE`. Le quote possono quindi
differire di poco da quelle dei log completi.

**Avvertenze.** Un solo seed. Climatologia e soglie sono stimate sugli anni di test: nella versione finale
saranno stimate sulla validazione.

In [ ]:
from pathlib import Path
import json, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from IPython.display import display

ROOT = Path(os.environ['PHYSIQ_ROOT']) if os.environ.get('PHYSIQ_ROOT') else next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'physiq_pv').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Apri il notebook dalla cartella del repository physiq_pv')
OUT = ROOT / 'outputs'
RUN = OUT / 'stgan_convgru_fullgrid_recent16_cyclic_bf16_cleanD_wdaily_seasonal_trend_ep3'
SCORES = {  # stesso periodo e stesse celle, tutti dopo la climatologia
    'modello': Path(f'{RUN}_gen_clim'),
    'distanza dal normale': Path(f'{RUN}_no_model_bal_clim'),
    'variazione in 3 h': Path(f'{RUN}_previous_bal_clim'),
}
LEAD_LOGS = ['anticipo_modello_tutti.log', 'anticipo_controlli.log', 'anticipo_modello_precedente_forti.log']
STRIDE = 29               # un istante ogni 29 (dispari: si leggono tutte le ore del giorno)
TOP_PERCENT = 5           # segnalato = nel 5% più alto dello score
SIMPLE_TOP_PERCENT = 5    # livello dei calcoli semplici per le aree (1 = solo le perturbazioni forti)
MIN_CELLS = 10            # perturbazioni dei calcoli semplici da almeno 10 celle (circa 25.000 km²)
CONTROL_DAYS = 30         # controllo del caso: segnalazioni del modello di circa un mese dopo
SAVE_FIGURES = False
FIGURE_DIR = OUT / 'figure_ep3'
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})
MODEL, NORMAL, CHANGE = SCORES

def show(figure, name):
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        figure.savefig(FIGURE_DIR / f'{name}.png', bbox_inches='tight', dpi=150)
    plt.show()

def manifest(directory):
    return json.loads((directory / 'metadata.json').read_text(encoding='utf-8'))

def open_scores(directory, spread=False):
    data = manifest(directory)
    file = (data.get('anomaly_std_file') if spread
            else data.get('anomaly_mean_file', data.get('scores_file', 'scores/anomaly_mean.npy')))
    return None if file is None or not (directory / file).is_file() else np.load(directory / file, mmap_mode='r')

def find_log(name):
    return next((path for path in (ROOT / name, OUT / name) if path.is_file()), None)

## 0. La run

Configurazione letta dall'ultimo checkpoint della run.

In [ ]:
checkpoints = sorted(RUN.glob('model_epoch_*.pt'), key=lambda path: int(re.search(r'_epoch_(\d+)', path.name).group(1)))
if checkpoints:
    import torch
    payload = torch.load(checkpoints[-1], map_location='cpu', weights_only=False)
    rates, model_config = payload.get('learning_rates') or {}, payload.get('model_config') or {}
    display(pd.Series({
        'epoche': payload.get('completed_epochs'), 'training': (payload.get('splits') or {}).get('train'),
        'lr G': rates.get('generator_learning_rate'), 'lr D': rates.get('discriminator_learning_rate'),
        'passi D:G': payload.get('discriminator_generator_update_ratio', '1:1'),
        'peso ricostruzione': payload.get('generator_reconstruction_weight'),
        'normalizzazione': (payload.get('normalization') or {}).get('kind'),
        'pesi delle variabili': 'sì' if model_config.get('reconstruction_feature_weights') is not None else 'no',
        'D vede la ricostruzione senza dropout': model_config.get('discriminator_input_dropout') is False,
        'campioni MC': (payload.get('mc_config') or {}).get('mc_samples'),
    }, name='valore').to_frame())
    del payload
else:
    print('Checkpoint non trovati in', RUN)

In [ ]:
# Lettura: ogni istante letto insieme a quello dopo (per la persistenza).
run_metadata = manifest(RUN)
grid = run_metadata['grid']
times_all = pd.to_datetime(np.load(SCORES[MODEL] / 'test_timestamps.npy'))
rows_read = np.arange(0, len(times_all) - 1, STRIDE)
times = times_all[rows_read]
value, following = {}, {}
for name, directory in SCORES.items():
    if not pd.to_datetime(np.load(directory / 'test_timestamps.npy')).equals(times_all):
        raise ValueError(f'{name}: non è sugli stessi istanti del modello')
    scores = open_scores(directory)
    value[name] = np.asarray(scores[rows_read], dtype=np.float32)
    following[name] = np.asarray(scores[rows_read + 1], dtype=np.float32)
    del scores
finite = np.logical_and.reduce([np.isfinite(values) for values in value.values()])
limit = {name: float(np.percentile(values[finite], 100 - TOP_PERCENT)) for name, values in value.items()}
flag = {name: (values > limit[name]) & finite for name, values in value.items()}
flag_next = {name: following[name] > limit[name] for name in SCORES}
M, N, C = flag[MODEL], flag[NORMAL], flag[CHANGE]
latitudes, longitudes = np.asarray(grid['latitudes']), np.asarray(grid['longitudes'])
cell_rows, cell_cols = np.asarray(grid['rows']), np.asarray(grid['cols'])
cell_latitude = latitudes[cell_rows]
print(f'{len(times)} istanti letti, dal {times[0].date()} al {times[-1].date()}; {M.shape[1]} celle')
print('soglie:', {name: round(level, 3) for name, level in limit.items()})

## 1. Cosa segnala il modello rispetto ai calcoli semplici

A sinistra, i valori segnalati dal modello divisi per chi altro li segnala. La parte "solo modello" è quella
che i calcoli semplici, nemmeno insieme, trovano. A destra, il contrario: quanta parte di ciò che segnala
ciascun calcolo semplice è segnalata anche dal modello.

In [ ]:
GROUPS = {
    'solo modello': M & ~N & ~C,
    'modello + distanza dal normale': M & N & ~C,
    'modello + variazione in 3 h': M & C & ~N,
    'modello + entrambi': M & N & C,
    'solo calcoli semplici': ~M & (N | C),
    'nessuno (riferimento)': ~M & ~N & ~C & finite,
}
MODEL_GROUPS = list(GROUPS)[:4]
COLORS = dict(zip(GROUPS, ['tab:red', 'tab:green', 'tab:orange', 'tab:purple', 'tab:blue', 'tab:grey']))
of_model = pd.Series({group: GROUPS[group].sum() / M.sum() for group in MODEL_GROUPS})
seen_by_model = pd.Series({
    'distanza dal normale': (N & M).sum() / N.sum(),
    'variazione in 3 h': (C & M).sum() / C.sum(),
    'almeno uno dei due': ((N | C) & M).sum() / (N | C).sum(),
})
figure, axes = plt.subplots(1, 2, figsize=(13, 3.6), constrained_layout=True)
axes[0].barh(of_model.index[::-1], 100 * of_model.values[::-1], color=[COLORS[g] for g in of_model.index[::-1]])
axes[0].set(xlabel='% dei valori segnalati dal modello', xlim=(0, 100), title='Chi altro segnala i valori del modello')
axes[1].barh(seen_by_model.index[::-1], 100 * seen_by_model.values[::-1], color='tab:blue')
axes[1].set(xlabel='% segnalato anche dal modello', xlim=(0, 100), title='Quanto dei calcoli semplici vede il modello')
for axis, series in zip(axes, (of_model[::-1], seen_by_model[::-1])):
    for position, share in enumerate(series.values):
        axis.text(100 * share + 1, position, f'{100 * share:.1f}%', va='center', fontsize=8)
show(figure, 'sovrapposizione')
display(pd.concat({'dei valori del modello': of_model, 'dei calcoli semplici visti dal modello': seen_by_model})
        .to_frame('quota').style.format('{:.1%}'))

## 2. Che tipi di evento sono

Per ogni gruppo:
- **posizione negli altri score**: il percentile mediano in ciascuno score. Un valore al 94° percentile di
  un calcolo semplice è appena sotto la sua soglia, uno al 50° è tutt'altro;
- **persistenza**: quanti sono segnalati di nuovo dal modello 3 ore dopo, nella stessa cella;
- **stagione** e **latitudine**;
- **di cosa è fatto l'errore del modello**: la quota di ogni variabile nell'errore pesato, come lo vede la
  perdita del modello.

In [ ]:
def percentile_of(name, mask):
    reference = np.sort(value[name][finite])
    return 100 * np.searchsorted(reference, value[name][mask]) / len(reference)

season_of = np.array(['inverno', 'inverno', 'primavera', 'primavera', 'primavera', 'estate', 'estate', 'estate',
                      'autunno', 'autunno', 'autunno', 'inverno'])[times.month.to_numpy() - 1]
rows = []
for group, mask in GROUPS.items():
    count = mask.sum()
    seasons = {season: (mask & (season_of == season)[:, None]).sum() / count
               for season in ('inverno', 'primavera', 'estate', 'autunno')}
    rows.append({'gruppo': group, 'quota di tutti i valori': count / finite.sum(),
                 **{f'percentile in {name}': np.median(percentile_of(name, mask)) for name in SCORES},
                 'di nuovo dal modello 3 h dopo': (flag_next[MODEL] & mask).sum() / count,
                 'latitudine mediana': float(np.median(np.broadcast_to(cell_latitude, mask.shape)[mask])),
                 **seasons})
groups_table = pd.DataFrame(rows).set_index('gruppo')
display(groups_table.style.format({column: '{:.1%}' for column in groups_table if column.startswith(('quota', 'di nuovo'))
                                   or column in ('inverno', 'primavera', 'estate', 'autunno')}
                                  | {column: '{:.0f}' for column in groups_table if column.startswith('percentile')}
                                  | {'latitudine mediana': '{:.1f}'}))
figure, axes = plt.subplots(1, 2, figsize=(13, 3.8), constrained_layout=True)
(groups_table[[f'percentile in {name}' for name in SCORES]].rename(columns=lambda c: c[len('percentile in '):])
 .plot.barh(ax=axes[0]))
axes[0].axvline(100 - TOP_PERCENT, color='grey', linestyle=':')
axes[0].set(xlabel='percentile mediano', title='Dove stanno i gruppi in ciascuno score'); axes[0].invert_yaxis()
(100 * groups_table[['inverno', 'primavera', 'estate', 'autunno']]).plot.barh(stacked=True, ax=axes[1], colormap='coolwarm')
axes[1].set(xlabel='% del gruppo', title='Stagione'); axes[1].invert_yaxis()
axes[1].legend(fontsize=8, loc='lower right')
show(figure, 'tipi_di_evento')

In [ ]:
feature_file = RUN / 'scores' / 'feature_scores.npy'
weights = (run_metadata.get('backend') or {}).get('reconstruction_feature_weights')
names = (run_metadata.get('preparation') or {}).get('features') or (list(weights) if weights else None)
if feature_file.is_file() and names:
    errors = np.load(feature_file, mmap_mode='r')
    weight = np.array([weights[name] for name in names]) if weights else np.ones(len(names))
    sums = {group: np.zeros(len(names)) for group in GROUPS}
    for start in range(0, len(rows_read), 8):
        block = np.asarray(errors[rows_read[start:start + 8]], dtype=np.float64) * weight
        usable = np.isfinite(block).all(axis=-1)
        for group, mask in GROUPS.items():
            sums[group] += block[mask[start:start + 8] & usable].sum(axis=0)
    del errors
    composition = pd.DataFrame({group: total / total.sum() for group, total in sums.items()}, index=names).T
    order = composition.loc['solo modello'].sort_values(ascending=False).index
    main = list(order[:7])
    shown = composition[main].assign(altre=composition.drop(columns=main).sum(axis=1))
    figure, axis = plt.subplots(figsize=(11, 4), constrained_layout=True)
    (100 * shown).plot.barh(stacked=True, ax=axis, colormap='tab10')
    axis.set(xlabel="% dell'errore pesato del modello", title="Di cosa è fatto l'errore del modello, per gruppo")
    axis.invert_yaxis(); axis.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    show(figure, 'errore_per_variabile')
    display(composition[order].style.format('{:.1%}'))
else:
    print('Errori per variabile non disponibili:', feature_file)

## 3. L'incertezza MC Dropout nei diversi tipi di evento

Per ogni valore, lo score del modello è la media dei 20 campioni MC Dropout; lo **spread** è la loro
deviazione standard. Se lo spread è piccolo, i campioni sono d'accordo. Una segnalazione è **solida** se resta
sopra la soglia anche abbassando lo score di uno spread: in quel caso non dipende dal dropout.

In [ ]:
spreads = open_scores(SCORES[MODEL], spread=True)
if spreads is not None:
    spread = np.asarray(spreads[rows_read], dtype=np.float32)
    del spreads
    usable = finite & np.isfinite(spread)
    rows = []
    for group, mask in GROUPS.items():
        chosen = mask & usable
        row = {'gruppo': group, 'spread mediano': float(np.median(spread[chosen]))}
        if group in MODEL_GROUPS:
            row['segnalazioni solide'] = float(((value[MODEL] - spread) > limit[MODEL])[chosen].mean())
            row['spread / distanza dalla soglia (mediana)'] = float(np.median(spread[chosen] / (value[MODEL][chosen] - limit[MODEL])))
        rows.append(row)
    dropout_table = pd.DataFrame(rows).set_index('gruppo')
    display(dropout_table.style.format({'spread mediano': '{:.3f}', 'segnalazioni solide': '{:.1%}',
                                        'spread / distanza dalla soglia (mediana)': '{:.2f}'}, na_rep='—'))
    figure, axes = plt.subplots(1, 3, figsize=(16, 3.8), constrained_layout=True)
    axes[0].barh(dropout_table.index[::-1], dropout_table['spread mediano'][::-1],
                 color=[COLORS[g] for g in dropout_table.index[::-1]])
    axes[0].set(title='Spread mediano dei campioni MC', xlabel='deviazione standard dello score')
    solid = dropout_table['segnalazioni solide'].dropna()
    axes[1].barh(solid.index[::-1], 100 * solid[::-1], color=[COLORS[g] for g in solid.index[::-1]])
    axes[1].set(title='Segnalazioni solide (sopra soglia anche −1 spread)', xlabel='%', xlim=(0, 100))
    sample = np.random.default_rng(0).choice(np.flatnonzero(usable.ravel()), size=min(400_000, usable.sum()), replace=False)
    shown = axes[2].hexbin(value[MODEL].ravel()[sample], spread.ravel()[sample], gridsize=80, bins='log', cmap='viridis')
    axes[2].axvline(limit[MODEL], color='red', linestyle='--', label='soglia')
    axes[2].set(xlabel='score del modello', ylabel='spread MC', title='Score e spread'); axes[2].legend()
    figure.colorbar(shown, ax=axes[2], label='valori (log)')
    show(figure, 'mc_dropout')
else:
    print('La run della climatologia non ha lo spread dei campioni MC')

## 4. Di quanto il modello allarga le perturbazioni dei calcoli semplici

A ogni istante, le celle segnalate da almeno un calcolo semplice formano delle **perturbazioni**: gruppi di
celle vicine, contando anche le diagonali. Si aggiungono poi le celle segnalate dal modello. Una perturbazione
si allarga se il modello segnala celle attaccate a essa.

**Allargamento** = area finale / area dei calcoli semplici: 1 vuol dire nessuna aggiunta, 2 vuol dire area
doppia. Le aree sono in km², con il coseno della latitudine.

**Controllo del caso**: lo stesso calcolo usando le segnalazioni del modello di circa `CONTROL_DAYS` giorni
dopo. Così si vede quanto allargamento viene solo dal fatto che il modello segnala il 5% delle celle.

In [ ]:
simple_limit = {name: float(np.percentile(value[name][finite], 100 - SIMPLE_TOP_PERCENT)) for name in (NORMAL, CHANGE)}
simple = ((value[NORMAL] > simple_limit[NORMAL]) | (value[CHANGE] > simple_limit[CHANGE])) & finite
height, width = len(latitudes), len(longitudes)
cell_km2 = (111.32 * float(grid.get('spacing', 0.5))) ** 2 * np.cos(np.deg2rad(cell_latitude))
STRUCTURE = np.ones((3, 3), dtype=bool)

def perturbations(model_flags, simple_flags):
    pieces = []
    for step in range(len(simple_flags)):
        s_map, m_map, area = np.zeros((height, width), bool), np.zeros((height, width), bool), np.zeros((height, width))
        s_map[cell_rows, cell_cols], m_map[cell_rows, cell_cols], area[cell_rows, cell_cols] = (
            simple_flags[step], model_flags[step], cell_km2)
        labels, count = ndimage.label(s_map | m_map, structure=STRUCTURE)
        if not count:
            continue
        labels = labels.ravel()
        total = lambda weights: np.bincount(labels, weights=weights.ravel(), minlength=count + 1)[1:]
        pieces.append(np.column_stack((np.full(count, step), total(s_map.astype(float)), total(s_map * area),
                                       total((s_map | m_map) * area), total((m_map & ~s_map) * area))))
    return pd.DataFrame(np.vstack(pieces), columns=['istante', 'celle semplici', 'area semplici', 'area finale',
                                                    'area solo modello'])

shift = max(1, round(CONTROL_DAYS * 24 / (3 * STRIDE)))  # istanti letti in circa CONTROL_DAYS giorni
found = {'modello': perturbations(M, simple), 'caso (modello di un mese dopo)': perturbations(np.roll(M, -shift, axis=0), simple)}
summary = {}
for label, table in found.items():
    big = table[table['celle semplici'] >= MIN_CELLS]
    ratio = big['area finale'] / big['area semplici']
    added = table['area solo modello']
    summary[label] = {
        'perturbazioni (≥ MIN_CELLS celle)': len(big),
        'area mediana dei calcoli semplici (km²)': big['area semplici'].median(),
        'allargamento mediano': ratio.median(),
        'allargamento complessivo (somma aree)': big['area finale'].sum() / big['area semplici'].sum(),
        'perturbazioni allargate almeno del 50%': (ratio >= 1.5).mean(),
        'area solo modello attaccata a una perturbazione': added[table['celle semplici'] > 0].sum() / added.sum(),
    }
summary = pd.DataFrame(summary)
display(summary.style.format('{:,.2f}'))
figure, axes = plt.subplots(1, 2, figsize=(13, 3.8), constrained_layout=True)
bins = np.geomspace(1, 20, 40)
for (label, table), color in zip(found.items(), ('tab:red', 'tab:grey')):
    big = table[table['celle semplici'] >= MIN_CELLS]
    axes[0].hist((big['area finale'] / big['area semplici']).clip(upper=20), bins=bins, alpha=0.6, color=color, label=label)
axes[0].set(xscale='log', xlabel='allargamento (area finale / area dei calcoli semplici)', ylabel='perturbazioni',
            title='Quanto si allargano le perturbazioni'); axes[0].legend()
size_bins = [MIN_CELLS, 30, 100, 300, np.inf]
for (label, table), color, offset in zip(found.items(), ('tab:red', 'tab:grey'), (-0.2, 0.2)):
    big = table[table['celle semplici'] >= MIN_CELLS]
    by_size = (big['area finale'] / big['area semplici']).groupby(pd.cut(big['celle semplici'], size_bins, right=False),
                                                                   observed=False).median()
    axes[1].bar(np.arange(len(by_size)) + offset, by_size.values, width=0.4, color=color, label=label)
axes[1].set_xticks(range(len(size_bins) - 1), [f'{a:.0f}–{b:.0f}' if np.isfinite(b) else f'≥{a:.0f}'
                                               for a, b in zip(size_bins[:-1], size_bins[1:])])
axes[1].axhline(1, color='black', linewidth=0.8)
axes[1].set(xlabel='celle della perturbazione dei calcoli semplici', ylabel='allargamento mediano',
            title='Per dimensione della perturbazione'); axes[1].legend()
show(figure, 'allargamento_aree')

## 5. Di quanto anticipa

Tabelle di `scripts/check_era5_lead.py`, calcolate su tutti gli istanti e le celle (log `anticipo_*.log`).
Per ogni inizio di un episodio dello score della riga, nella stessa cella, la tabella dice se l'altro score
era **già acceso**, e da quante ore. Le fasce colorate dividono il "già acceso" per numero di ore. Il pallino
nero è il caso: la stessa misura in istanti in cui lo score della riga è quieto e non comincia nulla.

In [ ]:
COLUMNS = ['starts', 'already', '3h', '6h', '9-12h', '15-24h', '>24h', 'lead med h', 'gap', 'together',
           'later', 'lag med h', 'none']
BANDS = ['3h', '6h', '9-12h', '15-24h', '>24h']
blocks = []
for name in LEAD_LOGS:
    path = find_log(name)
    if path is None:
        print('Log non trovato:', name)
        continue
    for line in path.read_text(encoding='utf-8', errors='replace').splitlines():
        if line.startswith('[lead]'):
            blocks.append({'log': name, 'levels': line.split('|')[1].strip(), 'rows': {}})
        elif blocks and line.startswith(('starts of', '  chance')):
            values = line[34:].split()
            if len(values) == len(COLUMNS):
                blocks[-1]['rows'][line[:34].strip()] = [float(value) for value in values]
bars = []
for block in blocks:
    table = pd.DataFrame(block['rows'], index=COLUMNS).T
    print(f"{block['log']} | {block['levels']}")
    display(table.round(1))
    labels = list(table.index)
    for index, label in enumerate(labels):
        if label.startswith('starts of'):
            chance = labels[index + 1] if index + 1 < len(labels) and labels[index + 1].startswith('chance') else None
            bars.append({'riga': f"{label} ({block['levels'].replace('flagged: the highest ', '')})",
                         **table.loc[label, BANDS].to_dict(),
                         'caso': table.loc[chance, 'already'] if chance else np.nan,
                         'mediana h': table.loc[label, 'lead med h']})
if bars:
    bars = pd.DataFrame(bars).set_index('riga')
    figure, axis = plt.subplots(figsize=(11, 0.45 * len(bars) + 1.5), constrained_layout=True)
    bars[BANDS].plot.barh(stacked=True, ax=axis, colormap='YlOrRd', width=0.7)
    axis.scatter(bars['caso'], np.arange(len(bars)), color='black', zorder=3, label='caso')
    axis.set(xlabel="% degli inizi di episodio in cui l'altro score era già acceso", ylabel='')
    axis.set_yticks(range(len(bars)), bars.index, fontsize=7)
    axis.invert_yaxis(); axis.legend(title='acceso da', fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    show(figure, 'anticipo')

## 6. In sintesi

I numeri principali, calcolati sopra.

In [ ]:
recap = {
    'valori del modello che nessun calcolo semplice segnala': f"{of_model['solo modello']:.1%}",
    'calcoli semplici (almeno uno) visti anche dal modello': f"{seen_by_model['almeno uno dei due']:.1%}",
}
if 'dropout_table' in globals():
    recap.update({f'segnalazioni solide, {group}': f"{dropout_table.loc[group, 'segnalazioni solide']:.1%}"
                  for group in MODEL_GROUPS})
recap.update({
    'allargamento complessivo delle perturbazioni (modello)':
        f"{summary.loc['allargamento complessivo (somma aree)', 'modello']:.2f}",
    'allargamento complessivo (caso)':
        f"{summary.loc['allargamento complessivo (somma aree)', 'caso (modello di un mese dopo)']:.2f}",
})
display(pd.Series(recap, name='valore').to_frame())

**Come leggerli**
- Il confronto è con calcoli che non usano un modello. Quello che segnala solo il modello va guardato nella
  sezione 2: di cosa è fatto, quando succede, quanto dura.
- L'allargamento delle aree conta solo se supera quello del controllo del caso.
- L'anticipo è rispetto agli altri rilevatori, non una previsione: ERA5 arriva con qualche giorno di ritardo.
- Un solo seed, e soglie e climatologia stimate sugli anni di test.